In [1]:
from genericbank.spark import get_spark
from genericbank import config

spark = get_spark("exploration")
spark

26/09/28 18:15:40 WARN Utils: Your hostname, TomaszMPC resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/28 18:15:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/tomasz/projects/genericbank-analytics-pyspark/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/tomasz/.ivy2/cache
The jars for the packages stored in: /home/tomasz/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6f2ab8f3-87c4-4a63-a957-4351dc86ba6f;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 230ms :: artifacts dl 9ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.3 from central in [default]
	io.delta#delta-storage;3.3.3 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0 

In [2]:
txn_raw = (
    spark.read
    .option("header", True)
    .csv(str(config.LANDING_DIR / "transactions"))
)
txn_raw.printSchema()


root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_ts: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- direction: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- counterparty_country: string (nullable = true)



In [3]:
txn_raw.show(5, truncate=False)
txn_raw.count()

+--------------+----------+-------------------+--------+---------+--------+-------+-----------------+--------------------+
|transaction_id|account_id|txn_ts             |amount  |direction|txn_type|channel|merchant_category|counterparty_country|
+--------------+----------+-------------------+--------+---------+--------+-------+-----------------+--------------------+
|T0000000029   |A00057758 |2025-10-01 06:00:30|6844,07 |CREDIT   |SALARY  |WEB    |NULL             |PL                  |
|T0000000044   |A00003915 |2025-10-01 06:00:44|3959.38 |CREDIT   |SALARY  |WEB    |NULL             |PL                  |
|T0000000047   |A00082374 |2025-10-01 06:00:45|6003.36 |CREDIT   |SALARY  |WEB    |NULL             |PL                  |
|T0000000059   |A00047888 |2025-10-01 06:00:59|11517.76|CREDIT   |SALARY  |WEB    |NULL             |PL                  |
|T0000000077   |A00077372 |2025-10-01 06:01:13|2878.18 |CREDIT   |SALARY  |WEB    |NULL             |PL                  |
+--------------+

4905630

In [4]:
from pyspark.sql import functions as F
(
    txn_raw
    .groupBy("channel")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+-------+-------+
|channel|count  |
+-------+-------+
|POS    |2093278|
|WEB    |1393444|
|MOBILE |869484 |
|ATM    |346752 |
|BRANCH |55306  |
| Pos   |16273  |
|pos    |16107  |
|pos    |16079  |
|Pos    |16071  |
|Web    |10815  |
|web    |10803  |
|web    |10762  |
| Web   |10692  |
|mobile |6911   |
| Mobile|6910   |
|Mobile |6742   |
|mobile |6732   |
|Atm    |2748   |
| Atm   |2722   |
|atm    |2682   |
+-------+-------+
only showing top 20 rows



In [5]:
dupes = (
    txn_raw
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
)
dupes.count()

24427

In [6]:
print("empty amount:     ", txn_raw.filter(F.col("amount").isNull()).count())
print("comma decimal:    ", txn_raw.filter(F.col("amount").contains(",")).count())
print("non-ISO timestamp:", txn_raw.filter(~F.col("txn_ts").rlike(r"^\d{4}-")).count())

empty amount:      4902


comma decimal:     49407


non-ISO timestamp: 98544


In [7]:
from genericbank import bronze

txn_bronze = bronze.add_ingestion_metadata(bronze.read_transactions(spark), run_id = "data exploration notebook")
txn_bronze.printSchema()
txn_bronze.select("transaction_id", "_source_file", "_ingested_at", "_run_id").show(3, truncate=False)

root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_ts: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- direction: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- counterparty_country: string (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _run_id: string (nullable = false)

+--------------+-------------------------------------------------------------------------------------------------+--------------------------+-------------------------+
|transaction_id|_source_file                                                                                     |_ingested_at              |_run_id                  |
+--------------+-------------------------------------------------------------------------------------------------+--------------------------

In [8]:
from genericbank import bronze, silver

txn_bronze = spark.read.format("delta").load(str(config.BRONZE_DIR / "transactions"))
txn = silver.parse_transactions(txn_bronze)

txn.printSchema()
print("null txn_ts:", txn.filter(F.col("txn_ts").isNull()).count())
print("null amount:", txn.filter(F.col("amount").isNull()).count())
txn.groupBy("channel").count().show()
txn.select("*").show(3, truncate=False)

root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- txn_ts_raw: string (nullable = true)
 |-- amount_raw: string (nullable = true)
 |-- direction: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- counterparty_country: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- txn_ts: timestamp (nullable = true)
 |-- amount: decimal(18,2) (nullable = true)



26/09/28 18:16:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


null txn_ts: 0


null amount: 4902
+-------+-------+
|channel|  count|
+-------+-------+
|    POS|2157808|
|    ATM| 357569|
| MOBILE| 896779|
| BRANCH|  56958|
|    WEB|1436516|
+-------+-------+

+--------------+----------+-------------------+----------+---------+------------+-------+-----------------+--------------------+-------------------------------------------------------------------------------------------------+--------------------------+--------------------------------+-------------------+------+
|transaction_id|account_id|txn_ts_raw         |amount_raw|direction|txn_type    |channel|merchant_category|counterparty_country|_source_file                                                                                     |_ingested_at              |_run_id                         |txn_ts             |amount|
+--------------+----------+-------------------+----------+---------+------------+-------+-----------------+--------------------+---------------------------------------------------------------

In [9]:
from genericbank import silver

txn_bronze = spark.read.format("delta").load(str(config.BRONZE_DIR / "transactions"))
txn = silver.parse_transactions(txn_bronze)
txn_dedup = silver.deduplicate_transactions(txn)

before, after = txn.count(), txn_dedup.count()
print(f"before: {before:,}  after: {after:,}  removed: {before - after:,}")

# sanity check: no id may appear twice after deduplication
txn_dedup.groupBy("transaction_id").count().filter(F.col("count") > 1).count()

before: 4,905,630  after: 4,881,203  removed: 24,427


0

In [10]:
acc_bronze = spark.read.format("delta").load(str(config.BRONZE_DIR / "accounts"))
flagged = silver.flag_rejects(txn_dedup, acc_bronze)

flagged.groupBy("reject_reason").count().show()
flagged.explain()

+---------------+-------+
|  reject_reason|  count|
+---------------+-------+
|UNKNOWN_ACCOUNT|   4790|
|           NULL|4871533|
| INVALID_AMOUNT|   4880|
+---------------+-------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [account_id#1871, transaction_id#1858, txn_ts_raw#1831, amount_raw#1845, direction#1913, txn_type#1928, channel#1943, merchant_category#1958, counterparty_country#1973, _source_file#1816, _ingested_at#1817, _run_id#1818, txn_ts#1884, amount#1898, CASE WHEN isnull(amount#1898) THEN INVALID_AMOUNT WHEN isnull(txn_ts#1884) THEN INVALID_TIMESTAMP WHEN isnull(_account_known#2342) THEN UNKNOWN_ACCOUNT END AS reject_reason#2360]
   +- BroadcastHashJoin [account_id#1871], [account_id#2339], LeftOuter, BuildRight, false
      :- Project [transaction_id#1858, account_id#1871, txn_ts_raw#1831, amount_raw#1845, direction#1913, txn_type#1928, channel#1943, merchant_category#1958, counterparty_country#1973, _source_file#1816, _ingested_at#1817, _run_id#1

In [11]:
from genericbank import silver

cust = silver.clean_customers(spark.read.format("delta").load(str(config.BRONZE_DIR / "customers")))
acc = silver.clean_accounts(spark.read.format("delta").load(str(config.BRONZE_DIR / "accounts")))

cust.printSchema()
cust.groupBy("segment").count().show()
acc.groupBy("account_type").count().show()

root
 |-- customer_id: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- relationship_start: date (nullable = true)

+-------+-----+
|segment|count|
+-------+-----+
| RETAIL|45112|
|PREMIER| 4888|
+-------+-----+

+------------+-----+
|account_type|count|
+------------+-----+
|     SAVINGS|20056|
|     CURRENT|50000|
|        CARD|25114|
+------------+-----+



In [ ]:
from genericbank import silver

def load(name):
    return spark.read.format("delta").load(str(config.BRONZE_DIR / name))

cust = silver.clean_customers(load("customers"))
acc = silver.clean_accounts(load("accounts"))
txn = silver.deduplicate_transactions(silver.parse_transactions(load("transactions")))
flagged = silver.flag_rejects(txn, acc)

txn_valid = flagged.filter(F.col("reject_reason").isNull())
silver_txn = silver.enrich_transactions(txn_valid, acc, cust)

print("rows:", silver_txn.count())
print("null customer_id:", silver_txn.filter(F.col("customer_id").isNull()).count())
silver_txn.groupBy("txn_month").count().orderBy("txn_month").show()
silver_txn.show(3, truncate=False)